## Data Cleaning Notebook

This is where we ingest files from `data/raw/`, process them appropriately, and store the processed versions in `data/processed/`.<br>
The functions live in `analysis/data_processing.py`. To regenerate everything without the notebook, run `python analysis/data_processing.py` from the project root.

What's changed: we are no longer looking at the GTFS dataset (https://511.org/open-data/transit) or "Housing Element APR Table A2" dataset (https://data.ca.gov). Our data is scoped more narrowly to three major datasets: `parcels.geojson`, `tod_zones.geojson`, and `epc_data.geojson`.

### Modules to Import (from `analysis/`)

In [10]:
import sys
sys.path.append("..")

from analysis import data_processing as dp
from analysis import visualize as viz

### Ingesting / Inspecting Raw Data

In [13]:
epc_data, tod_data, parcels_data = dp.load_raw()

In [14]:
parcels_data.to_parquet(dp.PROC_DATA_PATH / "parcels_tod.parquet") # .parquet is faster to load

In [15]:
dp.inspect(epc_data, "Equity Priority Community")
dp.inspect(parcels_data, "Parcels")
dp.inspect(tod_data, "Transit-Oriented Development Zones")

Inspecting Equity Priority Community dataset ...
(353, 37)
Index(['OBJECTID', 'tract_geoid', 'county_fip', 'tot_pop_poc', 'tot_pop_se',
       'tot_pop_po', 'tot_pop_ci', 'tot_pop_ov', 'tot_hh', 'tot_fam',
       'pop_poc', 'pop_over75', 'pop_spfam', 'pop_lep', 'pop_below2',
       'pop_disabi', 'pop_hus_re', 'pop_zvhhs', 'pct_poc', 'pct_over75',
       'pct_spfam', 'pct_lep', 'pct_below2', 'pct_disab', 'pct_hus_re',
       'pct_zvhhs', 'poc_1_2', 'over75_1_2', 'spfam_1_2', 'lep_1_2',
       'disab_1_2', 'below2_1_2', 'hus_re_1_2', 'zvhh_1_2', 'epc_2050p',
       'epc_class', 'geometry'],
      dtype='object')


,OBJECTID,tract_geoid,county_fip,tot_pop_poc,tot_pop_se,tot_pop_po,tot_pop_ci,tot_pop_ov,tot_hh,tot_fam,...,over75_1_2,spfam_1_2,lep_1_2,disab_1_2,below2_1_2,hus_re_1_2,zvhh_1_2,epc_2050p,epc_class,geometry
0,6,06075017602,75,3199,3199,3199,3189,3131,1594,524,...,0,0,0,1,1,1,1,1,Higher,"POLYGON ((-122.41094 37.77921, -122.40871 37.7..."


Inspecting Parcels dataset ...
(71742, 7)
Index(['APN', 'SitusCity', 'Land', 'Imps', 'LatestDocu', 'UseCode',
       'geometry'],
      dtype='object')


,APN,SitusCity,Land,Imps,LatestDocu,UseCode,geometry
0,1-115-12,OAKLAND,12312,0,2020,3000,"POLYGON ((-200581.412 -21587.574, -200587.916 ..."


Inspecting Transit-Oriented Development Zones dataset ...
(665, 4)
Index(['OBJECTID', 'zone_id', 'zone_label', 'geometry'], dtype='object')


,OBJECTID,zone_id,zone_label,geometry
0,1,764ca928-8bd7-4c6e-83d2-835dca264eed,Tier 1 - 200ft,"POLYGON Z ((-122.41005 37.62829 0, -122.41011 ..."


### Filtering and Joining

Main things to look out for: 
- valid "use codes" for parcels
- tracking parcels that lie in TOD (Transit-Oriented Development) Zones
- tracking TOD parcels that are in an EPC (Equity Priority Community) Area

In [16]:
epc_data = epc_data.to_crs(dp.CRS)
tod_data = tod_data.to_crs(dp.CRS)

In [17]:
parcels_data = dp.clean_parcels(parcels_data)
parcels_data.shape

(47512, 7)


In [19]:
epc_data = dp.filter_epc(epc_data)
epc_data.shape

(170, 37)

In [20]:
tod_data.zone_label.unique()

array(['Tier 1 - 200ft', 'Tier 1 - Half Mile', 'Tier 1 - Quarter Mile',
       'Tier 2 - 200ft', 'Tier 2 - Half Mile', 'Tier 2 - Quarter Mile'],
      dtype=object)

In [21]:
parcels_tod = dp.join_tod(parcels_data, tod_data)

In [23]:
parcels_tod = dp.join_epc(parcels_tod, epc_data)

In [24]:
parcels_tod = dp.add_features(parcels_tod)

### Finalize our Dataset!

In [25]:
dp.inspect(parcels_tod, "Parcels + TOD Zones") # we can see that we're able to reduce our parcel set by ~40% of rows

Inspecting Parcels + TOD Zones dataset ...
(43130, 19)
Index(['APN', 'SitusCity', 'Land', 'Imps', 'LatestDocu', 'UseCode', 'geometry',
       'acres', 'zone_id', 'zone_label', 'du_per_acre', 'max_units',
       'epc_class', 'is_epc', 'il_ratio', 'soft_site', 'soft_units',
       'recent_activity', 'tier'],
      dtype='object')


,APN,SitusCity,Land,Imps,LatestDocu,UseCode,geometry,acres,zone_id,zone_label,du_per_acre,max_units,epc_class,is_epc,il_ratio,soft_site,soft_units,recent_activity,tier
4,1-121-12,OAKLAND,348168,812392,2018.0,8100,"POLYGON ((-200475.892 -21660.385, -200468.822 ...",0.086101,dfbc6562-3685-40f1-bceb-ef878f866567,Tier 1 - Half Mile,100,8,Not EPC,False,2.333333,False,0,True,Tier 1


In [26]:
dp.summarize(parcels_tod)

total allowed units: 688068
share in an epc: 0.503
zone_label
Tier 1 - Half Mile       18369
Tier 2 - Half Mile       10308
Tier 2 - Quarter Mile     8449
Tier 1 - Quarter Mile     5559
Tier 2 - 200ft             366
Tier 1 - 200ft              79
Name: count, dtype: int64


### Visualizing our Processed Dataset

This map scopes to Alameda County only. We plan to scope to Santa Clara County in our final project submission.

In [29]:
viz.plot_density_map(parcels_tod, epc_data, tod_data);

### Save to `processed_data`
We use `.parquet` for efficient loading.

In [28]:
parcels_tod.to_parquet(dp.PROC_DATA_PATH / "parcels_tod_clean.parquet")
print('loaded cleaned data')

loaded cleaned data
